# Inverse design quickstart

This notebook will get users up and running with a very simple inverse design optimization with `tidy3d`. Inverse design uses the "adjoint method" to compute gradients of a figure of merit with respect to design parameters using only 2 simulations no matter how many design parameters are present. This gradient is then used to do high dimensional, gradient-based optimization of the system.

The setup we'll demonstrate here involves a point dipole source and a point field monitor on either side of a dielectric box. Using the adjoint plugin in `tidy3d`, we use gradient-based optimization to maximize the intensity enhancement at the measurement spot with respect to the box size in all 3 dimensions.

<img src="img/Adjoint_Quickstart.png" width="300" alt="Schematic of the design problem.">

For more detailed notebooks, see these

* [Tidy3D Autograd Tutorial](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd1Intro/).

* [Topology Optimization](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd3InverseDesign/).

* [Shape Optimization](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd5BoundaryGradients/).

* [Grating Coupler Inverse Design](https://www.flexcompute.com/tidy3d/examples/notebooks/Autograd6GratingCoupler/).


In [1]:
# To install other packages needed, uncomment lines below.
# !pip install optax

In [ ]:
import tidy3d as td
from tidy3d.web import run
import matplotlib.pylab as plt

import autograd as ag
import autograd.numpy as anp
import optax
from tidy3d.web.core.environment import Env, dev, prod
import tidy3d.web as web
import numpy as np

Env.set_current(dev)
web.configure("")

Configured successfully.


In [3]:
size_box =2.
center_box = 1.
vertices = np.array([(0,0), (1,0), (1,1)])
td.PolySlab(vertices=vertices, axis=2, slab_bounds=(-1, 1)).scaled(x=size_box, y=size_box, z=size_box).translated(x=center_box, y = 0., z = 0.)

PolySlab(attrs={}, type='PolySlab', axis=2, sidewall_angle=0.0, reference_plane='middle', slab_bounds=(-2.0, 2.0), dilation=0.0, vertices=array([[1., 0.],
       [3., 0.],
       [3., 2.]]))

In [4]:
size_box =2.
center_box = 1.
td.Sphere(center=(1,2,3), radius=2).scaled(x=size_box, y=size_box, z=size_box).translated(x=center_box, y = 0., z = 0.)

Transformed(attrs={}, type='Transformed', geometry=Sphere(attrs={}, type='Sphere', radius=2.0, center=(1.0, 2.0, 3.0)), transform=array([[2., 0., 0., 1.],
       [0., 2., 0., 0.],
       [0., 0., 2., 0.],
       [0., 0., 0., 1.]]))

## Setup

First, we set up some basic parameters and "static" components of our simulation.

In [5]:
# wavelength and frequency
wavelength = 1.55
freq0 = td.C_0 / wavelength

# permittivity of box
eps_box = 2

# size of sim in x,y,z
L = 10 * wavelength

# spc between sources, monitors, and PML / box
buffer = 1.0 * wavelength

In [6]:
# create a source to the left of sim
source = td.PointDipole(
    center=(-L / 2 + buffer, 0, 0),
    source_time=td.GaussianPulse(freq0=freq0, fwidth=freq0 / 10.0),
    polarization="Ez",
)

In [7]:
# create a monitor to right of sim for measuring intensity
monitor = td.FieldMonitor(
    center=(+L / 2 - buffer, 0, 0),
    size=(0.0, 0.0, 0.0),
    freqs=[freq0],
    name="point",
)

In [8]:
# create "base" simulation (the box will be added inside of the objective function later)
sim = td.Simulation(
    size=(L, L, L),
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=25),
    structures=[],
    sources=[source],
    monitors=[monitor],
    run_time=120 / freq0,
)

## Define objective function

Now we construct our objective function out of some helper functions. Our objective function measures the intensity enhancement at the measurement point as a function of a design parameter that controls the box size.

In [9]:
# function to get box size (um) as a function of the design parameter (-inf, inf)

size_min = 0
size_max = L - 4 * buffer

trans_min, trans_max = -L / 4, L / 4


def get_size(param: float):
    """Size of box as function of parameter, smoothly maps (-inf, inf) to (size_min, size_max)."""
    param_01 = 0.5 * (anp.tanh(param) + 1)
    return (size_max * param_01) + (size_min * (1 - param_01))

def get_translation(param_translate: float):
    """Maps a parameter to translation in x-direction."""
    param_01 = 0.5 * (anp.tanh(param_translate) + 1)
    return trans_min + (trans_max - trans_min) * param_01

# Transformed Geometry

In [10]:
def make_sim(param_size: float, param_translate: float):
    """Constructs simulation with box size and translation."""

    if param_size is None:
        return sim.copy()

    size_box = get_size(param_size)
    center_box = get_translation(param_translate)
    
    box = td.Structure(
        geometry=td.Box(center=(0., 0., 0.), size=(1., 1., 1.)).scaled(x=size_box, y=size_box, z=size_box).translated(x=center_box, y = 0., z = 0.),
        medium=td.Medium(permittivity=eps_box),
    )
    return sim.updated_copy(structures=[box])

# Box Geometry

In [11]:
# def make_sim(param_size: float, param_translate: float):
#     """Constructs simulation with box size and translation."""

#     if param_size is None:
#         return sim.copy()

#     size_box = get_size(param_size)
#     center_box = (get_translation(param_translate), get_translation(param_translate), get_translation(param_translate))
    
#     box = td.Structure(
#         geometry=td.Box(center=center_box, size=(size_box, size_box, size_box)),
#         medium=td.Medium(permittivity=eps_box),
#     )
#     return sim.updated_copy(structures=[box])

In [12]:
# function to compute and measure intensity as function of the design paramater


def measure_intensity(sim_data: td.SimulationData) -> float:
    """get intensity from SimulationData."""
    return anp.sum(sim_data.get_intensity(monitor.name).values)


def intensity(param_size: float, param_translate: float) -> float:
    """Intensity measured at monitor as function of parameter."""

    # make the sim using the parameter value
    sim_with_square = make_sim(param_size, param_translate)

    # run sim through tidy3d web API
    data = run(sim_with_square, task_name="inverse_design", verbose=False, local_gradient=True)

    # evaluate the intensity at the measurement position
    return measure_intensity(data)

In [13]:
# get the intensity with no box, for normalization (care about enhancement, not abs value)
intensity_norm = intensity(param_size=None, param_translate=None)
print(f"With no box, intensity = {intensity_norm:.4f}.")
print("This value will be used for normalization of the objective function.")

13:30:31 EST ERROR: Error running task                                          
             fdve-68d3143d-1880-4435-940a-217a7dde3c4d! Error message could not 
             be obtained, please contact customer support.                      

WebError: Error running task fdve-68d3143d-1880-4435-940a-217a7dde3c4d! Error message could not be obtained, please contact customer support.

In [ ]:
def objective_fn(params):
    """Objective function: maximize intensity at monitor."""
    return intensity(params[0], params[1]) / intensity_norm

## Optimization Loop

Next, we use `autograd` to construct a function that returns the gradient of our objective function and use this to run our gradient-based optimization in a for loop.

In [ ]:
# use autograd to get function that returns objective function and its gradient
val_and_grad_fn = ag.value_and_grad(objective_fn)

In [ ]:
# hyperparameters
num_steps = 4
learning_rate = 0.05

# initialize adam optimizer with starting parameter
params = anp.array([ -0.5, 0.0])
optimizer = optax.adam(learning_rate=learning_rate)
opt_state = optimizer.init(params)

# store history
objective_history = []  # the normalized objective function with no box
param_history = [params.copy()]  # -100 is approximately "no box" (size=0)

for i in range(num_steps):
    # compute gradient and current objective funciton value
    value, gradient = val_and_grad_fn(params)
    gradient = anp.array(gradient, dtype=float)
    # compute and apply updates to the optimizer based on gradient (-1 sign to maximize obj_fn)
    updates, opt_state = optimizer.update(-gradient, opt_state, params)
    params = optax.apply_updates(params, updates)
    params = anp.array(params, dtype=float)
    objective_history.append(value)
    param_history.append(params.copy())

    # outputs
    print(f"Step {i+1}: size={get_size(params[0]):.4f}, trans={get_translation(params[1]):.4f}, intensity={value:.4f}")

## Analysis
Finally we plot our results: optimization progress, field pattern, and box size vs intensity enhancement.

In [ ]:
# objective function vs iteration number
plt.plot(objective_history)
plt.xlabel("iteration number")
plt.ylabel("intensity enhancement (unitless)")
plt.title("intensity enhancement during optimization")
plt.show()

In [ ]:
# construct simulation with final parameters
sim_final = make_sim(param_size=param_history[-1][0], param_translate=param_history[-1][1])

# add a field monitor for plotting
fld_mnt = td.FieldMonitor(
    center=(+L / 2 - buffer, 0, 0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    name="fields",
)
sim_final = sim_final.updated_copy(monitors=[monitor, fld_mnt])

# run simulation
data_final = run(sim_final, task_name="quickstart_final", verbose=False)

In [ ]:
# record final intensity
intensity_final = measure_intensity(data_final)
intensity_final_normalized = intensity_final / intensity_norm

objective_history.append(intensity_final_normalized)

In [ ]:
# plot intensity distribution
ax = data_final.plot_field(
    field_monitor_name="fields", field_name="E", val="abs^2", vmax=intensity_final
)

ax.plot(source.center[0], 0, marker="o", mfc="limegreen", mec="black", ms=10)
ax.plot(monitor.center[0], 0, marker="o", mfc="orange", mec="black", ms=10)
plt.show()

In [ ]:
# scatter the intensity enhancement vs the box size
sizes = [get_size(p) for p in param_history]
objective_history = objective_history
_ = plt.scatter(sizes, objective_history)
ax = plt.gca()
ax.set_xlabel("box size (um)")
ax.set_ylabel("intensity enhancement (unitless)")
plt.title("intensity enhancement vs. box size")
plt.show()

In [ ]:
import numpy as np
import xarray as xr
import logging

log = logging.getLogger(__name__)

# Dummy helper for get_static (assumes inputs are already static)
def get_static(x):
    return x

def integrate_within_bounds(arr: xr.DataArray, dims: list[str], bounds) -> xr.DataArray:
    """
    Integrate a DataArray within bounds, assuming bounds are [2, N] for N dims.
    This function clips the coordinate values to the given bounds and then
    integrates using the trapezoidal rule.
    """
    # order bounds with dimension first (N, 2)
    bounds = np.asarray(bounds).T
    all_coords = {}

    # loop over all dimensions
    for dim, (bmin, bmax) in zip(dims, bounds):
        bmin = get_static(bmin)
        bmax = get_static(bmax)

        coord_values = np.copy(arr.coords[dim].data)

        # reset all coordinates outside of bounds to the bounds, so that dL = 0 in integral
        np.clip(coord_values, bmin, bmax, out=coord_values)

        all_coords[dim] = coord_values

    _arr = arr.assign_coords(**all_coords)

    # determine which dims have more than one point and integrate over them
    dims_integrate = [dim for dim in dims if len(_arr.coords[dim]) > 1]
    return _arr.integrate(coord=dims_integrate)

def compute_tangential_vectors(normal: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Given a normal vector, compute two orthogonal unit vectors perpendicular to it.
    """
    n = normal / np.linalg.norm(normal)
    # Pick an arbitrary reference that is not colinear with n.
    ref = np.array([1.0, 0.0, 0.0])
    if np.allclose(n, ref) or np.allclose(n, -ref):
        ref = np.array([0.0, 1.0, 0.0])
    t1 = np.cross(n, ref)
    if np.linalg.norm(t1) < 1e-6:
        ref = np.array([0.0, 1.0, 0.0])
        t1 = np.cross(n, ref)
    t1 = t1 / np.linalg.norm(t1)
    t2 = np.cross(n, t1)
    t2 = t2 / np.linalg.norm(t2)
    return t1, t2

class RotatedBox:
    def __init__(self, bounds, is_ccw=False):
        """
        bounds: tuple of ((xmin,xmax), (ymin,ymax), (zmin,zmax))
        is_ccw: flag for counterclockwise ordering (if needed)
        """
        self.bounds = bounds  # global bounds
        self.is_ccw = is_ccw

    def pop_axis(self, arr, axis: str):
        """
        For a numpy array representing bounds of shape (3,2) or
        for a list of dimension names, returns the entry for the given axis.
        For bounds, axis 'x' returns arr[0] and the others.
        """
        axis_map = {'x': 0, 'y': 1, 'z': 2}
        idx = axis_map[axis]
        dims = ['x', 'y', 'z']
        dims_perp = dims.copy()
        dims_perp.pop(idx)
        if isinstance(arr, np.ndarray):
            return arr[idx], arr[[axis_map[d] for d in dims_perp]]
        elif isinstance(arr, (list, tuple)):
            # When arr is a list of dimension names
            val = arr[idx]
            others = list(arr)
            others.pop(idx)
            return val, others
        else:
            raise ValueError("pop_axis not implemented for type " + str(type(arr)))

    def derivative_face_rotated(self, min_max_index: int, axis_normal: str,
                                  derivative_info, normal_vector: np.ndarray = None) -> float:
        """
        Compute the derivative (VJP) with respect to shifting a face.
        
        If 'normal_vector' is provided, it is taken as the rotated face normal.
        Otherwise, the face is assumed to be aligned with the canonical axis given by axis_normal.
        
        Parameters:
            min_max_index: 0 for lower face, 1 for upper face in the local coordinate.
            axis_normal: canonical axis that defines the face in the unrotated state (e.g. 'x').
            derivative_info: an object holding field and permittivity data.
            normal_vector: optional 3-element array specifying the face normal.
        
        Returns:
            The real-valued derivative computed by integrating field contributions over the face.
        """
        # 1. Determine the local basis.
        if normal_vector is None:
            # Default behavior: use canonical basis.
            if axis_normal == 'x':
                n_local = np.array([1.0, 0.0, 0.0])
                t1_local = np.array([0.0, 1.0, 0.0])
                t2_local = np.array([0.0, 0.0, 1.0])
            elif axis_normal == 'y':
                n_local = np.array([0.0, 1.0, 0.0])
                t1_local = np.array([1.0, 0.0, 0.0])
                t2_local = np.array([0.0, 0.0, 1.0])
            elif axis_normal == 'z':
                n_local = np.array([0.0, 0.0, 1.0])
                t1_local = np.array([1.0, 0.0, 0.0])
                t2_local = np.array([0.0, 1.0, 0.0])
            else:
                raise ValueError("Invalid axis_normal")
        else:
            # Use the given rotated normal (ensure unit length) and compute tangential vectors.
            n_local = normal_vector / np.linalg.norm(normal_vector)
            t1_local, t2_local = compute_tangential_vectors(n_local)

        # 2. Determine the face location and tangential integration bounds.
        # Get the 8 corners of the global box.
        bounds_global = np.array(self.bounds)  # shape (3,2)
        corners = np.array([[bounds_global[0, i], bounds_global[1, j], bounds_global[2, k]]
                            for i in (0, 1) for j in (0, 1) for k in (0, 1)])
        # Project corners onto the face normal.
        proj = corners.dot(n_local)
        if min_max_index == 0:
            coord_normal_face = np.min(proj)
        else:
            coord_normal_face = np.max(proj)

        # Transform corners into local coordinates (s, t1, t2)
        local_corners = np.zeros((8, 3))
        for i in range(8):
            local_corners[i, 0] = corners[i].dot(n_local)    # s coordinate (normal)
            local_corners[i, 1] = corners[i].dot(t1_local)     # first tangential coordinate
            local_corners[i, 2] = corners[i].dot(t2_local)     # second tangential coordinate

        # Determine integration bounds in the tangential directions.
        tol = 1e-6
        face_corners = local_corners[np.abs(local_corners[:, 0] - coord_normal_face) < tol]
        if face_corners.size == 0:
            face_corners = local_corners  # fallback if no corner exactly lies on the face
        t1_bounds = (face_corners[:, 1].min(), face_corners[:, 1].max())
        t2_bounds = (face_corners[:, 2].min(), face_corners[:, 2].max())
        bounds_perp = [t1_bounds, t2_bounds]  # expected shape: [ [min, max], [min, max] ]

        # 3. Define a helper to interpolate a global field onto the face
        # and assign local tangential coordinates.
        def interpolate_field_local(arr: xr.DataArray, target_value: float) -> xr.DataArray:
            # Extract global coordinates.
            X = arr.coords['x'].values
            Y = arr.coords['y'].values
            Z = arr.coords['z'].values
            Xg, Yg, Zg = np.meshgrid(X, Y, Z, indexing='ij')
            # Compute local coordinates: s (along normal), t1, and t2.
            s = n_local[0] * Xg + n_local[1] * Yg + n_local[2] * Zg
            t1 = t1_local[0] * Xg + t1_local[1] * Yg + t1_local[2] * Zg
            t2 = t2_local[0] * Xg + t2_local[1] * Yg + t2_local[2] * Zg
            s_da = xr.DataArray(s, dims=arr.dims, coords=arr.coords)
            t1_da = xr.DataArray(t1, dims=arr.dims, coords=arr.coords)
            t2_da = xr.DataArray(t2, dims=arr.dims, coords=arr.coords)
            arr_local = arr.assign_coords(s=s_da, t1=t1_da, t2=t2_da)
            # Interpolate along the new coordinate 's' to get values at the face.
            arr_at_face = arr_local.interp(s=target_value, assume_sorted=True)
            return arr_at_face

        # 4. Project the derivative field data onto the local basis.
        # (Assumes derivative_info.D_der_map and derivative_info.E_der_map contain
        #  DataArrays for "Ex", "Ey", and "Ez".)
        DEx = derivative_info.D_der_map["Ex"]
        DEy = derivative_info.D_der_map["Ey"]
        DEz = derivative_info.D_der_map["Ez"]
        D_normal_local = n_local[0] * DEx + n_local[1] * DEy + n_local[2] * DEz

        EEx = derivative_info.E_der_map["Ex"]
        EEy = derivative_info.E_der_map["Ey"]
        EEz = derivative_info.E_der_map["Ez"]
        E_perp1_local = t1_local[0] * EEx + t1_local[1] * EEy + t1_local[2] * EEz
        E_perp2_local = t2_local[0] * EEx + t2_local[1] * EEy + t2_local[2] * EEz

        # 5. Interpolate the projected fields onto the face (s = coord_normal_face).
        D_normal_face = interpolate_field_local(D_normal_local, coord_normal_face)
        E_perp1_face = interpolate_field_local(E_perp1_local, coord_normal_face)
        E_perp2_face = interpolate_field_local(E_perp2_local, coord_normal_face)

        # 6. Use the existing integration helper to integrate over the tangential dims.
        # In our local coordinate system, these dimensions are 't1' and 't2'.
        integral_D = integrate_within_bounds(arr=D_normal_face, dims=["t1", "t2"], bounds=bounds_perp)
        integral_E1 = integrate_within_bounds(arr=E_perp1_face, dims=["t1", "t2"], bounds=bounds_perp)
        integral_E2 = integrate_within_bounds(arr=E_perp2_face, dims=["t1", "t2"], bounds=bounds_perp)

        # 7. Get permittivity contrasts (assumed scalar values).
        delta_eps_inv_normal = 1.0 / derivative_info.eps_in - 1.0 / derivative_info.eps_out
        delta_eps = derivative_info.eps_in - derivative_info.eps_out

        # 8. Combine the contributions.
        # Multiply the integrated D contribution by -delta_eps_inv_normal,
        # and each E contribution by delta_eps.
        vjp_value = (-delta_eps_inv_normal * integral_D +
                     delta_eps * integral_E1 +
                     delta_eps * integral_E2)
        return np.real(vjp_value)

# ----------------------------------------------------------------------
# Dummy classes and test code for demonstration.
# ----------------------------------------------------------------------
class DerivativeInfo:
    def __init__(self, bounds, D_der_map, E_der_map, eps_in, eps_out):
        self.bounds = bounds
        self.D_der_map = D_der_map
        self.E_der_map = E_der_map
        self.eps_in = eps_in
        self.eps_out = eps_out

if __name__ == "__main__":
    # Global box bounds: ((xmin,xmax), (ymin,ymax), (zmin,zmax))
    bounds = ((0, 10), (0, 5), (0, 3))
    
    # Create dummy xarray DataArrays for field derivatives.
    coords = {'x': np.linspace(0, 10, 11),
              'y': np.linspace(0, 5, 6),
              'z': np.linspace(0, 3, 4)}
    dims = ('x', 'y', 'z')
    Ex = xr.DataArray(np.ones((11, 6, 4)), dims=dims, coords=coords)
    Ey = xr.DataArray(2 * np.ones((11, 6, 4)), dims=dims, coords=coords)
    Ez = xr.DataArray(3 * np.ones((11, 6, 4)), dims=dims, coords=coords)
    
    D_der_map = {"Ex": Ex, "Ey": Ey, "Ez": Ez}
    E_der_map = {"Ex": Ex, "Ey": Ey, "Ez": Ez}
    
    eps_in = 2.0
    eps_out = 1.0
    derivative_info = DerivativeInfo(bounds, D_der_map, E_der_map, eps_in, eps_out)
    
    # Create an instance of the RotatedBox.
    box = RotatedBox(bounds)
    
    # Example 1: Use default (unrotated) behavior for an 'x' face.
    result_default = box.derivative_face_rotated(min_max_index=1,
                                                   axis_normal='x',
                                                   derivative_info=derivative_info,
                                                   normal_vector=None)
    print("Default (unrotated) face derivative:", result_default)
    
    # Example 2: Provide a rotated normal.
    rotated_normal = np.array([0.8, 0.6, 0.0])  # Example rotated normal (not yet normalized)
    result_rotated = box.derivative_face_rotated(min_max_index=1,
                                                   axis_normal='x',  # axis_normal is used only if normal_vector is None
                                                   derivative_info=derivative_info,
                                                   normal_vector=rotated_normal)
    print("Rotated face derivative:", result_rotated)
